# Chapter 18: Testing Your Website with Scrapers

In [38]:
import random
import re
import time
import unittest
from urllib.parse import unquote, urljoin

from bs4 import BeautifulSoup
import requests
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver import ActionChains

## Python `unittest`

In [13]:
class TestAddition(unittest.TestCase):

    def setUp(self):
        print("Setting up the test...")

    def tearDown(self):
        print("Tearing down the test.")

    def test_two_plus_two(self):
        total = 2 + 2
        self.assertEqual(4, total)


if __name__ == "__main__":
    unittest.main(argv=[""], exit=False)
    # %reset

.

Setting up the test...
Tearing down the test.


..
----------------------------------------------------------------------
Ran 3 tests in 0.901s

OK


Once deleted, variables cannot be recovered. Proceed (y/[n])?  n


Nothing done.


## Testing Wikipedia

In [15]:
class TestWikipedia(unittest.TestCase):
    bs = None
    def setUpClass():
        url = "https://en.wikipedia.org/wiki/Python_(programming_language)"
        resp = requests.get(url, headers={"User-Agent": "Mr.Robot-Linux_Machine"})
        TestWikipedia.bs = BeautifulSoup(resp.text, "lxml")

    def test_titleTest(self):
        pageTitle = TestWikipedia.bs.find("h1").get_text(strip=True)
        self.assertEqual("Python (programming language)", pageTitle)

    def test_contentExists(self):
        content = TestWikipedia.bs.find(
            "div", {"id": "mw-content-text"}
        )
        self.assertIsNotNone(content)


if __name__ == "__main__":
    unittest.main(argv=[""], exit=False)
    %reset

.

Setting up the test...
Tearing down the test.


..
----------------------------------------------------------------------
Ran 3 tests in 0.848s

OK


Once deleted, variables cannot be recovered. Proceed (y/[n])?  y


In [50]:
class TestWikipediaArticles(unittest.TestCase):

    def test_PageProperties(self):
        self.url = "https://en.wikipedia.org/wiki/Innovation"
        # Test the first 10 pages we encounter
        for i in range(1, 11):
            resp = requests.get(self.url, headers={"User-Agent": "Mr.Robot-Linux_Machine"})
            self.bs = BeautifulSoup(resp.text, "lxml")
            titles = self.titleMatchesURL()
            self.assertEqual(titles[0], titles[1])
            self.assertTrue(self.contentExists())
            self.url = self.getNextLink()
        print("Done!")

    def titleMatchesURL(self):
        pageTitle = self.bs.find("h1").get_text(strip=True)
        urlTitle = self.url[(self.url.index("/wiki/") + 6):]
        urlTitle = urlTitle.replace("_", " ")
        urlTitle = unquote(urlTitle)
        return (pageTitle.lower(), urlTitle.lower())

    def contentExists(self):
        content = self.bs.find("div", {"id": "mw-content-text"})
        if content is not None:
            return True
        return False

    def getNextLink(self):
        # Returns random link on the page
        links = self.bs.find("div", {"id": "bodyContent"}).find_all(
            "a",
            href = re.compile(r"^(?:https://en\.wikipedia\.org)?/wiki/[^:#]+$")
        )
        # print("ALL LINKS:", links)
        randomLink = random.SystemRandom().choice(links)
        return urljoin("https://en.wikipedia.org", randomLink.attrs["href"])


if __name__ == "__main__":
    unittest.main(argv=[""], exit=False)
    # %reset

.
----------------------------------------------------------------------
Ran 1 test in 9.064s

OK


Done!


## Testing with Selenium

In [15]:
chrome_options = Options()
chrome_options.add_argument("--headless")
driver = webdriver.Chrome(options=chrome_options)

driver.get("https://en.wikipedia.org/wiki/Innovation")
assert "Innovation" in driver.title
print("Full Title:", driver.title)

driver.close()

driver.quit()

Full Title: Innovation - Wikipedia


### Interacting with the Site

In [46]:
form_url = "https://pythonscraping.com/pages/files/form.html"

driver = webdriver.Chrome()
driver.get(form_url)
time.sleep(2)

first_name = driver.find_element(By.CSS_SELECTOR, "input[name=firstname]")
last_name = driver.find_element(By.CSS_SELECTOR, "input[name=lastname]")
submit_btn = driver.find_element(By.CSS_SELECTOR, "input[type=submit]")

# ===== METHOD 1: =====
# first_name.send_keys("John")
# last_name.send_keys("Will")
# submit_btn.click()

# ===== METHOD 2: =====
actions = ActionChains(driver).click(first_name) \
    .send_keys("Johnson") \
    .click(last_name) \
    .send_keys("Stevenson") \
    .send_keys(Keys.ENTER)
actions.perform()


time.sleep(2)
# driver.close()
driver.quit()

### Drag & drop

In [51]:
class TestDragAndDrop(unittest.TestCase):
    driver = None
    url = "https://pythonscraping.com/pages/javascript/draggableDemo.html"

    def setUp(self):
        self.driver = webdriver.Chrome()
        self.driver.get(self.url)
        time.sleep(2)

    def tearDown(self):
        driver.quit()

    def test_drag(self):
        element = self.driver.find_element(By.ID, "draggable")
        target = self.driver.find_element(By.ID, "div2")
        actions = ActionChains(self.driver)
        actions.drag_and_drop(element, target).perform()
        self.assertEqual(
            "You are definitely not a bot!",
            self.driver.find_element(By.ID, "message").text
        )


if __name__ == "__main__":
    unittest.main(argv=[""], exit=False)
    # %reset

.
----------------------------------------------------------------------
Ran 1 test in 4.149s

OK


### Taking screenshots

In [91]:
chrome_options = Options()
chrome_options.add_argument("--headless=new")
chrome_options.add_argument("--window-size=1920,1080")
driver = webdriver.Chrome(options=chrome_options)
# driver.maximize_window()

driver.get("https://pythonscraping.com/")
time.sleep(2)
driver.save_screenshot("./selenium-screenshot.png")
driver.quit()